In [194]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/assignment_release/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/assignment_release


In [195]:
# ------------------------
# Dummy baseline for Kaggle submission
# Generates random multi-label predictions
# ------------------------
import os
import csv
import random
from tqdm import tqdm

# --- Paths ---
TEST_DIR = "Amazon_products/test"  # modify if needed
TEST_CORPUS_PATH = os.path.join(TEST_DIR, "test_corpus.txt")  # product_id \t text
SUBMISSION_PATH = "submission.csv"  # output file

# --- Constants ---
NUM_CLASSES = 531  # total number of classes (0–530)
MIN_LABELS = 1     # minimum number of labels per sample
MAX_LABELS = 3     # maximum number of labels per sample

# --- Load test corpus ---
def load_corpus(path):
    """Load test corpus into {pid: text} dictionary."""
    pid2text = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) == 2:
                pid, text = parts
                pid2text[pid] = text
    return pid2text

pid2text_test = load_corpus(TEST_CORPUS_PATH)
pid_list_test = list(pid2text_test.keys())

# --- Generate random predictions ---
all_pids, all_labels = [], []
for pid in tqdm(pid_list_test, desc="Generating dummy predictions"):
    n_labels = random.randint(MIN_LABELS, MAX_LABELS)
    labels = random.sample(range(NUM_CLASSES), n_labels)
    labels = sorted(labels)
    all_pids.append(pid)
    all_labels.append(labels)

# --- Save submission file ---
with open(SUBMISSION_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["pid", "labels"])
    for pid, labels in zip(all_pids, all_labels):
        writer.writerow([pid, ",".join(map(str, labels))])

print(f"Dummy submission file saved to: {SUBMISSION_PATH}")
print(f"Total samples: {len(all_pids)}, Classes per sample: {MIN_LABELS}-{MAX_LABELS}")

Generating dummy predictions: 100%|██████████| 19658/19658 [00:00<00:00, 308548.74it/s]

Dummy submission file saved to: submission.csv
Total samples: 19658, Classes per sample: 1-3


여기부터 MAGIC START!

In [196]:
# Step 0. Imports & Paths
import os
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
from tqdm import tqdm

import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


In [197]:
# 프로젝트 루트 기준 경로 설정 (dummy_baseline와 같은 레벨에 Amazon_products가 있다고 가정)
ROOT = Path(".")          # 지금 노트북이 dummy_baseline/ 안에 있을 때
DATA_DIR = ROOT / "Amazon_products"

TRAIN_PATH = DATA_DIR / "train" / "train_corpus.txt"
TEST_PATH  = DATA_DIR / "test" / "test_corpus.txt"
CLASSES_PATH = DATA_DIR / "classes.txt"
HIER_PATH = DATA_DIR / "class_hierarchy.txt"
KEYWORDS_PATH = DATA_DIR / "class_related_keywords.txt"

OUT_DIR = Path(".")  # silver label, embedding 저장 위치 (dummy_baseline/)
OUT_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [198]:
# Step 1.1 Load class names, hierarchy, and keywords

def load_classes(path: Path):
    id2name = {}
    name2id = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cid_str, cname = line.split("\t", 1)
            cid = int(cid_str)
            cname = cname.strip()
            id2name[cid] = cname
            name2id[cname] = cid
    return id2name, name2id


def load_hierarchy(path: Path, num_classes: int):
    parents = defaultdict(list)
    children = defaultdict(list)
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            p_str, c_str = line.split("\t")
            p = int(p_str); c = int(c_str)
            parents[c].append(p)
            children[p].append(c)

    # depth 계산 (root는 parent가 없는 노드)
    depth = {cid: None for cid in range(num_classes)}
    roots = [cid for cid in range(num_classes) if len(parents[cid]) == 0]

    q = deque()
    for r in roots:
        depth[r] = 0
        q.append(r)

    while q:
        u = q.popleft()
        for v in children[u]:
            if depth[v] is None:
                depth[v] = depth[u] + 1
                q.append(v)

    return parents, children, depth, roots

def load_keywords(path: Path, name2id):
    id2keywords = defaultdict(list)
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # 콜론 없는 라인은 (예: 'swings') 그냥 스킵
            if ":" not in line:
                continue
            cname, kws_str = line.split(":", 1)
            cname = cname.strip()
            if cname not in name2id:
                # classes.txt에 없는 이름이면 건너뜀
                continue
            cid = name2id[cname]
            kw_list = [w.strip() for w in kws_str.split(",") if w.strip()]
            id2keywords[cid] = kw_list
    return id2keywords

id2name, name2id = load_classes(CLASSES_PATH)
num_classes = len(id2name)
print("num_classes:", num_classes)

id2keywords = load_keywords(KEYWORDS_PATH, name2id)
parents, children, depth, roots = load_hierarchy(HIER_PATH, num_classes)


print("roots:", roots[:10], "...")
print("example class 0:", id2name[0], id2keywords.get(0, []), "depth:", depth[0])


num_classes: 531
roots: [0, 3, 10, 23, 40, 169] ...
example class 0: grocery_gourmet_food ['snacks', 'condiments', 'beverages', 'specialty_foods', 'spices', 'cooking_oils', 'baking_ingredients', 'gourmet_chocolates', 'artisanal_cheeses', 'organic_foods'] depth: 0


In [199]:
# Step 1.2 Load train/test corpus

def load_corpus(path: Path):
    """train_corpus.txt / test_corpus.txt에서 (pid, text)를 줄 단위로 읽기"""
    pids, texts = [], []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            pid, txt = line.split("\t", 1)
            pids.append(pid)
            texts.append(txt)
    return pids, texts

train_pids, train_texts = load_corpus(TRAIN_PATH)
test_pids,  test_texts  = load_corpus(TEST_PATH)

print("train docs:", len(train_pids))
print("test docs:", len(test_pids))
print("sample doc:", train_pids[0], train_texts[0][:200], "...")


train docs: 29487
test docs: 19658
sample doc: 0 omron hem 790it automatic blood pressure monitor with advanced omron health management software so far this machine has worked well and is very simple to use . it is nice to have immediate feedback on ...


In [200]:
# Step 1.3 Build textual descriptions for each class (name + keywords)

def build_class_texts(id2name, id2keywords):
    texts = []
    for cid in range(len(id2name)):
        name = id2name[cid]
        kws = id2keywords.get(cid, [])
        text = name
        if kws:
            text += ": " + ", ".join(kws)
        texts.append(text)
    return texts

class_texts = build_class_texts(id2name, id2keywords)
print(class_texts[0])


grocery_gourmet_food: snacks, condiments, beverages, specialty_foods, spices, cooking_oils, baking_ingredients, gourmet_chocolates, artisanal_cheeses, organic_foods


In [201]:
# Step 1.4. Encode documents and classes with BERT (with optional better encoder)

import torch
from transformers import AutoTokenizer, AutoModel

# (선택) sentence-transformers가 있으면 더 좋은 문장 임베딩을 쓰고,
# 없으면 BERT CLS pooling으로 fallback
USE_SENTENCE_TRANSFORMERS = True

def get_text_encoder(device):
    if USE_SENTENCE_TRANSFORMERS:
        try:
            from sentence_transformers import SentenceTransformer
            model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
            return ("st", model)
        except Exception as e:
            print("[INFO] sentence-transformers not available -> fallback to BERT CLS. Reason:", str(e))

    MODEL_NAME = "bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    bert = AutoModel.from_pretrained(MODEL_NAME).to(device)
    bert.eval()
    return ("bert_cls", (tokenizer, bert))

ENCODER_TYPE, ENCODER = get_text_encoder(device)

@torch.no_grad()
def encode_texts(texts, batch_size=64, max_length=128):
    """
    texts: list[str] -> torch.FloatTensor [N, H]
    """
    if ENCODER_TYPE == "st":
        # sentence-transformers는 내부적으로 batch 처리 가능
        embs = ENCODER.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_tensor=True,
            normalize_embeddings=False,
        )
        return embs.detach().cpu()

    # BERT CLS pooling
    tokenizer, bert = ENCODER
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i : i + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)
        outputs = bert(**enc)
        cls = outputs.last_hidden_state[:, 0, :]   # [B, H]
        all_embs.append(cls.detach().cpu())
    return torch.cat(all_embs, dim=0)


In [202]:
# Stage1: train/class embeddings (no intermediate .pt cache)
import torch

print("[EMB] Computing embeddings in-memory (no .pt cache)...")
train_embs = encode_texts(train_texts, batch_size=64, max_length=128)
class_embs = encode_texts(class_texts, batch_size=128, max_length=64)

print("train_embs:", train_embs.shape)
print("class_embs:", class_embs.shape)
print("encoder:", ENCODER_TYPE)


[EMB] Computing embeddings in-memory (no .pt cache)...


Batches:   0%|          | 0/461 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

train_embs: torch.Size([29487, 384])
class_embs: torch.Size([531, 384])
encoder: st


In [203]:
# Step 1.5 Build EMBEDDING-based document–class similarity (cosine) (no .pt cache)

import torch
import torch.nn.functional as F

def cosine_sims(A, B, eps=1e-8):
    """
    A: [N, H], B: [C, H] -> sims [N, C]
    """
    A = F.normalize(A, dim=1, eps=eps)
    B = F.normalize(B, dim=1, eps=eps)
    return A @ B.T

print("[SIMS] Computing doc-class cosine similarities in-memory (no .pt cache)...")
sims = cosine_sims(train_embs.float(), class_embs.float()).cpu()  # [N, C]

print("sims shape:", tuple(sims.shape))
print("sample sims for doc 0 (top 5):")
top5 = torch.topk(sims[0], k=5)
for score, cid in zip(top5.values, top5.indices):
    print(float(score), int(cid), id2name[int(cid)])


[SIMS] Computing doc-class cosine similarities in-memory (no .pt cache)...
sims shape: (29487, 531)
sample sims for doc 0 (top 5):
0.3730110824108124 118 health_monitors
0.23474447429180145 526 hydrometers
0.23378878831863403 87 stress_reduction
0.2240990847349167 417 automatic_feeders
0.2210438996553421 333 test_kits


In [204]:
# Step 1.6 Silver label generation (embedding sims + hierarchy) (no .pt cache)

import torch

num_docs, num_classes = sims.shape
print("num_docs:", num_docs, "num_classes:", num_classes)

# ---------- ancestors precompute ----------
def build_ancestors(parents, num_classes):
    """
    parents:
    - dict[int -> int]  (single parent)
    - dict[int -> list[int]] (multi parent)
    - list/array where parents[c] is int or list[int]
    return:
    ancestors[c] = set of all ancestors of class c (excluding itself)
    """

    def get_parents(x):
        # parents[x]를 "항상 list[int]" 형태로 정규화
        if isinstance(parents, dict):
            p = parents.get(x, None)
        else:
            p = parent
        if p is None:
            return []
        if isinstance(p, (list, tuple, set)):
            return list(p)
        return [int(p)]

    ancestors = [set() for _ in range(num_classes)]
    for cid in range(num_classes):
        stack = get_parents(cid)
        visited = set()
        while stack:
            p = stack.pop()
            if p in visited:
                continue
            visited.add(p)
            ancestors[cid].add(p)
            stack.extend(get_parents(p))

    return ancestors

ancestors = build_ancestors(parents, num_classes)

# ---------- hyperparams (너가 쉽게 튜닝 가능) ----------
TOPK = 5                 # 문서당 상위 k개 클래스를 core로
POS_THR = 0.25           # cosine similarity threshold (0~1)
MIN_ANCESTOR_DEPTH = 1   # 조상 중 depth>=1인 애들만 추가

# depth 계산
depth = {}
def get_depth(c):
    if c in depth:
        return depth[c]
    ps = parents.get(c, []) if isinstance(parents, dict) else parents[c]
    if ps is None or ps == []:
        depth[c] = 0
        return 0
    if isinstance(ps, int):
        d = 1 + get_depth(ps)
    else:
        d = 1 + min(get_depth(p) for p in ps)
    depth[c] = d
    return d

for c in range(num_classes):
    get_depth(c)

# silver_labels: {-1 ignored, 0 negative, 1 positive}
silver_labels = torch.full((num_docs, num_classes), -1, dtype=torch.int8)

for i in range(num_docs):
    row = sims[i]  # [C]
    core = torch.topk(row, k=TOPK).indices.tolist()
    core = [cid for cid in core if float(row[cid]) >= POS_THR]

    if not core:
        continue

    pos_set = set()
    for cid in core:
        pos_set.add(cid)
        for anc in ancestors[cid]:
            if depth.get(anc, 0) >= MIN_ANCESTOR_DEPTH:
                pos_set.add(anc)

    silver_labels[i, list(pos_set)] = 1

silver_labels.shape


num_docs: 29487 num_classes: 531


torch.Size([29487, 531])

In [205]:
# Step 1.7 Save silver labels and some stats

torch.save(
    {
        "pids": train_pids,
        "silver_labels": silver_labels,  # int8 tensor, values in {1, 0, -1}
    },
    OUT_DIR / "train_silver_labels.pt",
)

# ---- 통계 출력 ----
num_pos = int((silver_labels == 1).sum().item())
num_neg = int((silver_labels == 0).sum().item())
num_ign = int((silver_labels == -1).sum().item())
print("Positive labels:", num_pos)
print("Negative labels:", num_neg)
print("Ignored labels:", num_ign)
print("Avg positives per doc:", num_pos / num_docs)

# 샘플 몇 개 찍어보기
for idx in range(10):
    text = train_texts[idx][:200].replace("\n", " ")
    y = silver_labels[idx]
    pos_cids = torch.nonzero(y == 1, as_tuple=False).view(-1).tolist()
    print("\nDoc:", train_pids[idx])
    print("Text:", text, "...")
    print("Silver classes:",
          [(cid, id2name[cid]) for cid in pos_cids])


Positive labels: 215379
Negative labels: 0
Ignored labels: 15442218
Avg positives per doc: 7.304201851663445

Doc: 0
Text: omron hem 790it automatic blood pressure monitor with advanced omron health management software so far this machine has worked well and is very simple to use . it is nice to have immediate feedback on ...
Silver classes: [(74, 'medical_supplies_equipment'), (118, 'health_monitors')]

Doc: 1
Text: natural factors whey factors chocolate works well , but there is a lot of dead space in the container when you first open it up . the container comes 3 4 4 5 full and the rest in empty space . ...
Silver classes: [(218, 'cooking_baking_supplies'), (265, 'candy_chocolate'), (269, 'chocolate'), (271, 'snack_food'), (276, 'chocolate_bars'), (456, 'granola_trail_mix_bars'), (490, 'chocolate_covered_fruit')]

Doc: 2
Text: clif bar builder 's bar , 2 . 4 ounce bars i love the peanut butter builder 's bars . while amazon is great for so many things , a trip to tj 's is too good t

2단계 시작!1

In [206]:
# Step 2.0: Silver labels 후처리 - shallow(depth 낮은) 라벨은 학습에서 제외

import torch

num_docs, num_classes = silver_labels.shape
print("silver_labels shape:", tuple(silver_labels.shape))

def stats(lbl):
    pos = int((lbl == 1).sum())
    neg = int((lbl == 0).sum())
    ign = int((lbl == -1).sum())
    return pos, neg, ign

pos_cnt, neg_cnt, ign_cnt = stats(silver_labels)
print(f"[Before] Positive: {pos_cnt}, Negative: {neg_cnt}, Ignored: {ign_cnt}")
print(f"[Before] Avg positives per doc: {pos_cnt / num_docs:.4f}")

# depth 딕셔너리 기반으로 shallow ids 추출
# (보통 depth 0~1은 root/부모급이라 제외)
SHALLOW_DEPTH_MAX = 1
shallow_ids = [cid for cid in range(num_classes) if depth.get(cid, 0) <= SHALLOW_DEPTH_MAX]

print("number of shallow (ignored) classes:", len(shallow_ids))

if len(shallow_ids) > 0:
    shallow_ids_tensor = torch.tensor(shallow_ids, dtype=torch.long)
    silver_labels[:, shallow_ids_tensor] = -1  # ignore

pos_cnt, neg_cnt, ign_cnt = stats(silver_labels)
print(f"[After]  Positive: {pos_cnt}, Negative: {neg_cnt}, Ignored: {ign_cnt}")
print(f"[After]  Avg positives per doc: {pos_cnt / num_docs:.4f}")


silver_labels shape: (29487, 531)
[Before] Positive: 215379, Negative: 0, Ignored: 15442218
[Before] Avg positives per doc: 7.3042
number of shallow (ignored) classes: 70
[After]  Positive: 103960, Negative: 0, Ignored: 15553637
[After]  Avg positives per doc: 3.5256


In [207]:
# GCN용 adjacency (양방향 + self-loop + D^{-1/2} A D^{-1/2})
A = torch.zeros(num_classes, num_classes, dtype=torch.float32)
for p in range(num_classes):
    for c in children[p]:
        A[p, c] = 1.0
        A[c, p] = 1.0

A = A + torch.eye(num_classes)  # self-loop

deg = A.sum(dim=1)
deg_inv_sqrt = torch.where(deg > 0, deg.pow(-0.5), torch.zeros_like(deg))
D_inv_sqrt = torch.diag(deg_inv_sqrt)
A_hat = D_inv_sqrt @ A @ D_inv_sqrt   # [C, C]

# device로 옮기기
A_hat = A_hat.to(device)
class_embs = class_embs.to(device)
train_embs = train_embs.to(device)
silver_labels = silver_labels.to(device)

A_hat.shape
A_hat


tensor([[0.0588, 0.0808, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0808, 0.1111, 0.2357,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.2357, 0.5000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.5000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.5000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.5000]],
       device='cuda:0')

In [208]:
# Step 2.2 Dataset & DataLoader

import torch
from torch.utils.data import Dataset, DataLoader

class SilverDataset(Dataset):
    def __init__(self, doc_embs, labels):
        self.doc_embs = doc_embs
        self.labels = labels  # [N, C], values in {1,0,-1}

    def __len__(self):
        return self.doc_embs.size(0)

    def __getitem__(self, idx):
        return self.doc_embs[idx], self.labels[idx]

# train/val split
N = train_embs.size(0)
val_ratio = 0.1
n_val = int(N * val_ratio)
n_train = N - n_val

perm = torch.randperm(N)
train_idx = perm[:n_train]
val_idx = perm[n_train:]

train_ds = SilverDataset(train_embs[train_idx], silver_labels[train_idx])
val_ds   = SilverDataset(train_embs[val_idx],   silver_labels[val_idx])

batch_size = 256
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, drop_last=False)

print("train_loader:", len(train_loader), "val_loader:", len(val_loader))


train_loader: 104 val_loader: 12


In [209]:
# Step 2.3
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, X, A_hat):
        return self.lin(A_hat @ X)

class LabelGCN(nn.Module):
    def __init__(self, num_classes, emb_dim, hidden_dim, A_hat, class_init, temperature=10.0, drop=0.2):
        super().__init__()
        self.num_classes = num_classes
        self.emb_dim = emb_dim
        self.temperature = temperature

        self.register_buffer("A_hat", A_hat)                 # [C,C]
        self.class_embed = nn.Parameter(class_init.clone())  # [C,H]

        self.gcn1 = GCNLayer(emb_dim, hidden_dim)
        self.gcn2 = GCNLayer(hidden_dim, emb_dim)

        self.doc_proj = nn.Linear(emb_dim, emb_dim)
        self.drop = nn.Dropout(drop)

    def get_class_embeddings(self):
        X = self.class_embed
        H = F.relu(self.gcn1(X, self.A_hat))
        H = self.drop(H)
        H = self.gcn2(H, self.A_hat)
        return H

    def forward(self, doc_embs):
        # doc side
        doc_h = F.relu(self.doc_proj(doc_embs))
        doc_h = self.drop(doc_h)

        # class side
        class_h = self.get_class_embeddings()

        # ✅ 핵심: cosine similarity logits (스케일 안정화)
        doc_h = F.normalize(doc_h, dim=1)
        class_h = F.normalize(class_h, dim=1)

        logits = self.temperature * (doc_h @ class_h.T)  # [B,C]
        return logits


In [210]:
# Step 2.4 Loss function & evaluation (full)

import torch
import torch.nn as nn

bce = nn.BCEWithLogitsLoss(reduction="none")

def multilabel_loss(logits, labels):
    """
    logits: [B, C]
    labels: [B, C] in {1,0,-1}; -1은 ignore
    """
    mask = (labels != -1).float()
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits.device, requires_grad=True)

    target = (labels == 1).float()
    loss_raw = bce(logits, target)         # [B, C]
    loss = (loss_raw * mask).sum() / mask.sum()
    return loss

@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total, n = 0.0, 0
    for X, Y in loader:
        X, Y = X.to(device), Y.to(device)
        logits = model(X)
        loss = multilabel_loss(logits, Y)
        total += float(loss.item())
        n += 1
    return total / max(n, 1)


In [211]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def refine_topk(model, doc_embs, base_labels, topk_pos=5, bottomk_neg=30, temp=10.0, bs=2048, device="cuda"):
    """
    반환: {-1,0,1}
    - doc당 topk_pos: +1
    - doc당 bottomk_neg: 0
    - 나머지: -1 (ignore)
    - base_labels==1은 항상 +1로 유지
    """
    model.eval()
    N, C = base_labels.shape
    out = torch.full((N, C), -1, dtype=torch.int8)          # 기본 ignore
    out[base_labels == 1] = 1                               # 기존 pos 유지

    for i in range(0, N, bs):
        X = doc_embs[i:i+bs].to(device)

        # ✅ cosine score로 스케일 안정화 (dot 폭발 방지)
        class_h = F.normalize(model.get_class_embeddings(), dim=1)  # [C,H]
        doc_h = F.normalize(F.relu(model.doc_proj(X)), dim=1)       # [B,H]
        score = doc_h @ class_h.T                                   # [-1,1]
        score = temp * score                                        # sharpness

        # base pos는 선택에서 제외(덮어쓰기 방지)
        fixed_pos = (base_labels[i:i+bs].to(device) == 1)
        score_for_pos = score.masked_fill(fixed_pos, -1e9)
        score_for_neg = score.masked_fill(fixed_pos,  1e9)

        pos_idx = torch.topk(score_for_pos, k=min(topk_pos, C), largest=True).indices
        neg_idx = torch.topk(score_for_neg, k=min(bottomk_neg, C), largest=False).indices

        chunk = out[i:i+bs].to(device)
        chunk.scatter_(1, pos_idx, 1)
        chunk.scatter_(1, neg_idx, 0)
        out[i:i+bs] = chunk.cpu()

    return out

def stats(x):
    pos = int((x==1).sum()); neg=int((x==0).sum()); ign=int((x==-1).sum())
    N,C=x.shape
    print(f"pos={pos} ({pos/(N*C):.2%})  neg={neg} ({neg/(N*C):.2%})  ign={ign} ({ign/(N*C):.2%})")
    print(f"avg pos/doc={pos/N:.2f}  avg neg/doc={neg/N:.2f}  avg ign/doc={ign/N:.2f}")

refined_labels = refine_topk(
    model,
    train_embs.cpu(),
    silver_labels.cpu(),
    topk_pos=5,
    bottomk_neg=30,
    temp=10.0,
    device=device
).to(device)

stats(refined_labels.cpu())


pos=251395 (1.61%)  neg=884610 (5.65%)  ign=14521592 (92.74%)
avg pos/doc=8.53  avg neg/doc=30.00  avg ign/doc=492.47


In [212]:
# =========================
# Stage 2 training (FIXED)
# =========================

def train_stage2(
    model,
    doc_embs,
    labels,          # ★ 반드시 refined_labels
    epochs=5,
    batch_size=256,
    lr=1e-3,
    device="cuda",
):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    N = doc_embs.size(0)
    idx = torch.randperm(N)

    for ep in range(1, epochs+1):
        total_loss = 0.0
        cnt = 0

        for i in range(0, N, batch_size):
            bidx = idx[i:i+batch_size]
            X = doc_embs[bidx].to(device)
            Y = labels[bidx].to(device)   # ★ refined_labels

            logits = model(X)

            # ignore(-1)는 loss에서 제외
            mask = (Y != -1)
            if mask.sum() == 0:
                continue

            target = (Y == 1).float()
            loss = F.binary_cross_entropy_with_logits(
                logits[mask],
                target[mask]
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()
            cnt += 1

        print(f"[Stage2][Epoch {ep}] loss={total_loss/max(cnt,1):.4f}")


In [213]:
# Step 2.6 Build datasets/loaders + init model + (IMPORTANT) TRAIN IT HERE

train_ds2 = SilverDataset(train_embs[train_idx], refined_labels[train_idx])
val_ds2   = SilverDataset(train_embs[val_idx],   refined_labels[val_idx])

train_loader2 = DataLoader(train_ds2, batch_size=batch_size, shuffle=True)
val_loader2   = DataLoader(val_ds2, batch_size=batch_size, shuffle=False)

emb_dim = int(train_embs.shape[1])  # Step 2.7 저장에서 씀

model = LabelGCN(
    num_classes=num_classes,
    emb_dim=emb_dim,
    hidden_dim=256,
    A_hat=A_hat,
    class_init=class_embs.to(device),
    temperature=10.0
).to(device)

# -------------------------
# ★★ 핵심: 여기서 학습을 실제로 수행해야 함
# -------------------------
EPOCHS2 = 5
LR2 = 1e-3

print("[Stage2] before training val_loss:", eval_loss(model, val_loader2))

train_stage2(
    model=model,
    doc_embs=train_embs[train_idx],
    labels=refined_labels[train_idx],
    epochs=EPOCHS2,
    batch_size=batch_size,
    lr=LR2,
    device=device,
)

print("[Stage2] after training val_loss:", eval_loss(model, val_loader2))


[Stage2] before training val_loss: 0.7804122567176819
[Stage2][Epoch 1] loss=0.1369
[Stage2][Epoch 2] loss=0.0487
[Stage2][Epoch 3] loss=0.0400
[Stage2][Epoch 4] loss=0.0362
[Stage2][Epoch 5] loss=0.0340
[Stage2] after training val_loss: 0.028151188821842272


In [214]:
# Step 2.7 Save final model (for step 3 prediction)

torch.save(
    {
        "state_dict": model.state_dict(),
        "A_hat": A_hat.cpu(),
        "class_init": class_embs.cpu(),
        "emb_dim": emb_dim,
        "num_classes": num_classes,
    },
    OUT_DIR / "labelgcn_final.pt",
)

print("Saved:", OUT_DIR / "labelgcn_final.pt")


Saved: labelgcn_final.pt


여기부터 3단계!!!

In [215]:
# Step 3.1 Encode test corpus with BERT

test_embs = encode_texts(test_texts, batch_size=32, max_length=128)
test_embs = test_embs.to(device)

print("test_embs:", test_embs.shape)


Batches:   0%|          | 0/615 [00:00<?, ?it/s]

test_embs: torch.Size([19658, 384])


In [216]:
# Step 3.2 Predict per-class probabilities for test docs

@torch.no_grad()
def predict_probs(model, doc_embs, batch_size=256):
    model.eval()
    all_probs = []
    for i in range(0, doc_embs.size(0), batch_size):
        X = doc_embs[i : i + batch_size].to(device)
        logits = model(X)
        probs = torch.sigmoid(logits).cpu()
        all_probs.append(probs)
    return torch.cat(all_probs, dim=0)

test_probs = predict_probs(model, test_embs)
print("test_probs:", test_probs.shape)


test_probs: torch.Size([19658, 531])


In [217]:
# Step 3.3 Convert probabilities to final labels  (GUARANTEE 2~3 labels)

import torch
from collections import defaultdict

N_test, C = test_probs.shape

# ancestors 없으면 한 번 계산 (부모가 list인 DAG도 처리)
def compute_ancestors(parents_dict, cid):
    anc = set()
    stack = list(parents_dict.get(cid, []))
    while stack:
        u = stack.pop()
        if u in anc or u == -1:
            continue
        anc.add(u)
        stack.extend(parents_dict.get(u, []))
    return anc

if "ancestors" not in globals():
    ancestors = {cid: compute_ancestors(parents, cid) for cid in range(num_classes)}

# leaf 노드 집합
leaf_ids = {cid for cid in range(num_classes) if len(children[cid]) == 0}

# 하이퍼파라미터(원래 쓰던 거 유지)
TOP_K_CAND = 30
MIN_DEPTH  = 1
MIN_LABELS = 2
MAX_LABELS = 3
MIN_PROB   = 0.25

def select_labels_for_doc(probs_1d):
    """
    probs_1d: torch.Tensor [C]
    반환: 길이 2~3의 int list (절대 빈 리스트 X)
    """
    # 1) 후보 랭킹 (상위 TOP_K_CAND)
    ranked = torch.topk(probs_1d, k=min(TOP_K_CAND, C)).indices.tolist()

    def greedy_pick(candidates, selected):
        for cid in candidates:
            cid = int(cid)
            # (a) 너무 상위(depth) 제외
            if depth.get(cid, 0) <= MIN_DEPTH:
                continue
            # (b) 확률 컷
            if float(probs_1d[cid]) < MIN_PROB:
                continue

            # (c) parent/child conflict: 더 구체적인 쪽만 남기기
            conflict = False
            for s in list(selected):
                if s in ancestors[cid]:
                    selected.remove(s)          # cid가 더 구체적 → s 제거
                elif cid in ancestors[s]:
                    conflict = True             # cid가 더 상위 → 버림
                    break
            if conflict:
                continue

            selected.append(cid)
            if len(selected) == MAX_LABELS:
                break
        return selected

    selected = []

    # 2) leaf 우선
    leaf_cand = [cid for cid in ranked if cid in leaf_ids]
    selected = greedy_pick(leaf_cand, selected)

    # 3) 부족하면 leaf 제약 완화
    if len(selected) < MIN_LABELS:
        selected = greedy_pick(ranked, selected)

    # 4) 그래도 부족하면(핵심!) 모든 제약을 풀고 top2/3 강제 채움
    if len(selected) < MIN_LABELS:
        all_ranked = torch.argsort(probs_1d, descending=True).tolist()
        for cid in all_ranked:
            cid = int(cid)
            if cid in selected:
                continue
            selected.append(cid)
            if len(selected) == MIN_LABELS:
                break

    # 5) 최대 3개로 정리 + 중복 제거
    selected = list(dict.fromkeys(selected))[:MAX_LABELS]

    # 마지막 안전장치: 진짜로 2개 미만이면 top2로
    if len(selected) < MIN_LABELS:
        selected = torch.topk(probs_1d, k=MIN_LABELS).indices.tolist()

    return selected

pred_labels_per_doc = [select_labels_for_doc(test_probs[i]) for i in range(N_test)]

# sanity check
lens = [len(x) for x in pred_labels_per_doc]
print("len distribution:", {k: lens.count(k) for k in sorted(set(lens))})
print("first 5:", list(zip(test_pids[:5], pred_labels_per_doc[:5])))


len distribution: {3: 19658}
first 5: [('0', [316, 117, 152]), ('1', [498, 194, 77]), ('2', [430, 178, 166]), ('3', [369, 455, 454]), ('4', [177, 219, 152])]


In [218]:
# Step 3.4 Build submission CSV (NO NULL GUARANTEE)

import csv

SUBMISSION_PATH = "2021320312_final.csv"

with open(SUBMISSION_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "label"])

    for pid, labels in zip(test_pids, pred_labels_per_doc):
        # 안전장치: 혹시라도 비면 top2로 채움(이제 거의 안 일어남)
        if labels is None or len(labels) < 2:
            probs = test_probs[test_pids.index(pid)]  # 비추천(느림)인데 거의 안 탈 거라 안전장치용
            labels = torch.topk(probs, k=2).indices.tolist()

        label_str = ",".join(map(str, labels))
        writer.writerow([pid, label_str])

print(f"Saved submission file to: {SUBMISSION_PATH}")


Saved submission file to: 2021320312_final.csv
